# Dunnhumby GAT · M1~M5_D 8모형 · seed49

교수님 미팅을 위한 **GAT 1시드 적용 비교**입니다. M2 표현·M3 그래프·M4 손실의 독립 효과와 결합 효과를 같은 GAT 기준모형과 비교합니다.

|순서|모형|M2 표현|M3 가중치|M4 손실|
|---|---|---|---|---|
|1|M1|없음|1|무가중|
|2|M2|적응형 N/V|1|무가중|
|3|M3|없음|중심화 N/V|무가중|
|4|M4|없음|1|고객 내 평균1|
|5|M5_A|적응형 N/V|중심화 N/V|무가중|
|6|M5_B|적응형 N/V|1|고객 내 평균1|
|7|M5_C|없음|중심화 N/V|고객 내 평균1|
|8|M5_D|적응형 N/V|중심화 N/V|고객 내 평균1|

## 고정 조건
- **L4 GPU**, seed49, train DAY1~704 / test DAY705~711, 신규 상품 추천(학습 구매쌍 정답·후보 제외), MIN_ITEM_INTER=1, uniform 음성1개.
- 각 모형 **300 epoch 마지막 체크포인트만 test 1회 평가**. validation·holdout·조기종료·중간test·자동 추가시드 없음. 결과를 보고 계수·epoch를 바꾸지 않습니다.
- ID64·전파2층64·single head. 이웃 attention은 LeakyReLU(.2), 집계 후 ELU, layer0/1/2 평균으로 최종64차원을 만듭니다. 기존 프로젝트 GAT의 자기연결 없음·dropout 없음·층 평균을 유지한 추천용 구현이며 원논문의 노드분류 벤치마크 정확재현이 아닙니다.
- batch8192, Adam lr.0005, ID L2 .001, M2 공유 L2 1e-6, GAT projection/attention L2 1e-6. 모든 표현과 attention을 **하나의 optimizer·추천손실로 공동학습**합니다. 사전학습 동결·외부재정렬 없음.
- M2는 확정 N/V·아이템 경제 속성·신뢰도·RBF·양성상품의 현재 구매자 제외·적응계수·직교결합을 유지합니다. 인코더 출력64차원. CLV는 학습구간365일 historical CLV proxy입니다.

## M3를 GAT 전파에 적용하는 방법
현재 중심화 N/V 계산에서 얻은 **차수 정규화 전의 원시 엣지 가중치** `w_ij`를 그대로 사용합니다.

`alpha_tilde_ij = alpha_ij * w_ij / sum_k(alpha_ik * w_ik)`

계산은 `softmax_j(LeakyReLU(attention logit_ij) + log(w_ij))`로 수행합니다. M3가 없는 네 모형은 w=1입니다. 이진 연결집합은 모든 모형에서 같으며 GAT용 별도 가중치/강도 탐색은 없습니다.

**적용 차이:** LightGCN/NGCF의 가중 차수 정규화와 동일하지 않습니다. 이웃별 재정규화로 고객별 공통 배율이 상쇄되므로 원래 M3의 질량 보존 성질을 GAT의 성질로 주장하지 않습니다. 원시 M3 가중치 진단과 GAT attention 행합·엔트로피를 따로 저장합니다.

## 저장·재개와 결과 해석
- 8모형을 순서대로 새 학습합니다. LightGCN/NGCF의 성능·checkpoint를 GAT 결과로 재사용하지 않습니다. 기존 M2/M4 결과는 입력 정합성 확인에만 사용합니다.
- 매 epoch 모델·Adam·numpy/torch/CUDA 난수를 Drive에 동기 저장합니다. 끊긴 경우 **같은 노트북·코드·참조/출력 경로로 위에서부터 재실행**하면 완료모형은 건너뛰고 미완료모형만 마지막 저장 epoch 다음부터 재개합니다. Colab 자동 재접속은 제공하지 않습니다.
- 손상 checkpoint나 Drive 저장 실패는 파일을 보존하고 중단합니다. 매 epoch 로컬 직렬화 후 Drive 복사·저장 완료를 확인합니다.
- 첫3 epoch의 첫5배치에 입력/음성추출·학습계산을 계측합니다. GAT 전체 자료의 L4 시간·최대메모리는 아직 실측하지 않았으며 NGCF 실행시간을 GAT 예상시간으로 쓰지 않습니다. 대용량 엣지 메시지는 청크로 집계해 전체 엣지×차원 중간행렬 저장을 피합니다.
- 전체142지표·저/중/고CLV·노출·모형간 비교·gradient/attention 진단·학습 및 저장시간을 ZIP으로 받습니다. 불리한 결과도 보존합니다.
- **1시드이며 SD·95%CI를 산출하지 않으므로 유의성을 주장하지 않습니다.** 약10시드 최종평균·CLV 배정귀속·새 독립확증을 대신하지 않습니다.

근거: [GAT 원논문](https://arxiv.org/abs/1710.10903), [GAT 연산 공식](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.GATConv.html). 이 노트북은 별도 PyG 설치 없이 프로젝트의 PyTorch 희소 GAT 집계를 사용합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. L4로 변경 후 실행하세요.'
SOURCE_COMMIT = '64d554a30a0c2f47b79545efaceda6f31967fee5'
REPO = Path('/content/clv-dh-gat-s49-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 확정 CLV 입력과 기존 참조 검증 — 학습 없음
기존 Dunnhumby M2 10시드 및 M4 10시드 폴더에서 원본 보고서의 해시·분할·입력·M2 특징·M3/M4 가중치 정합성을 확인합니다. 파일이 맞지 않으면 학습 전에 중단합니다. **이 단계에서 기존 모델을 새로 학습하지 않습니다.** 재사용 준비 모듈의 A/B 안내가 나타나더라도 실제 실행 목록은 아래 GAT 8개입니다.

기본 참조 폴더가 이동/변경되었다면 `m2_reference`, `m4_reference`에 보관 원본 ZIP 또는 폴더의 Drive 경로를 넣으세요. 첫 실행 이후에는 경로를 유지하세요. 출력은 별도 GAT 폴더입니다.

In [ ]:
import clv_dh_gat_final8_seed49 as screen
cfg = screen.configure(
    # m2_reference='/content/drive/MyDrive/원본M2결과.zip',
    # m4_reference='/content/drive/MyDrive/clv_m4_user_centered_final10_results.zip',
)
print('M2 입력 검증 참조:', cfg.m2_reference)
print('M3/M4 입력 검증 참조:', cfg.m4_reference)
print('GAT 새 결과:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('실제 새 학습:', list(screen.ORDER), '/ seed49 / 각300epoch')
print('진행·checkpoint·결과 저장:', prepared['run_dir'])

## 2. GAT 8개 모형 순차 학습
모형당 시작3개 epoch 및 이후5epoch마다 손실·전체 학습 시간·Drive 저장 시간을 표시합니다. 처음5배치가 끝나면 입력/음성추출과 학습계산의 배치당 표본 시간도 먼저 표시하므로 첫epoch가 끝나기 전에도 확인할 수 있습니다.

매epoch checkpoint를 Drive에 저장하고, 각 모형300epoch 완료 시 전체142지표를 평가·저장한 뒤 다음 모형으로 넘어갑니다. **로그 빈도와 무관하게 Drive 저장은 매epoch 실행**됩니다. 학습 중 다른 셀을 실행하거나 런타임을 바꾸지 마세요.

각 시간은 다음과 같습니다.
- 입력·음성추출: 같은 순서의 uniform 음성표본 생성과 배치 자료 준비/전송.
- 학습계산: 그래프 전파·BPR 손실·역전파·Adam 업데이트.
- 학습 전체: epoch의 입력·계산·로그/진행기록을 포함한 벽시계 시간.
- Drive 저장완료: 로컬 직렬화·복사/해시·Drive 저장/메타데이터 전체 시간.


In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 시간 기록 확인 — 학습·평가 없음
아래 셀은 저장된 시간 CSV만 읽습니다. 학습 셀이 끝났거나 중단된 뒤 실행하세요. 마지막10개 epoch를 보여주며 전체 기록은 결과 ZIP의 `timing/`에 포함됩니다. 입력 준비 총 시간은 `preparation_timing.json`에 있습니다. 표본 시간은 첫3개 epoch만 있으며, 나머지 빈칸은 미측정입니다.

In [ ]:
import pandas as pd
from IPython.display import display
timing_files = sorted((prepared['run_dir'] / 'progress' / 'timing').glob('*.csv'))
if timing_files:
    timing = pd.concat([pd.read_csv(p) for p in timing_files], ignore_index=True)
    columns = ['model_id', 'epoch', 'train_seconds', 'save_seconds',
               'sample_input_seconds_per_batch', 'sample_compute_seconds_per_batch',
               'checkpoint_local_seconds', 'checkpoint_drive_seconds', 'checkpoint_metadata_seconds']
    display(timing[columns].tail(10))
    print('단위: 초. 표본 계측은 추가 동기화 비용을 포함합니다. test 성능을 측정하지 않습니다.')
else:
    print('아직 완료 epoch의 시간 기록이 없습니다. 초기 배치 시간은 학습 출력에서 확인하세요.')

## 3. 저장된 완료 모형만 비교 — 학습·재평가 없음
8모형 절대지표와 **같은 GAT M1 대비** 변화율입니다. 미완료 모형은 빈칸이며0점이 아닙니다. 전체142지표·모든 모형 간 비교·개입 진단은 CSV에 보관됩니다. 단일 시드이므로 표준편차/신뢰구간을 만들지 않습니다.

세션이 끊겼다면1번 준비 셀까지 실행한 다음 이 셀로 바로 와도 됩니다. 전체 결과를 받으려면2번 학습 셀을 이어서 실행하세요.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(columns=screen.ORDER))
against_m1 = result['comparison'].query("reference == 'M1'")
if not against_m1.empty:
    display(against_m1.pivot(index='metric', columns='model', values='relative_change_pct')
            .reindex(index=metrics, columns=screen.ORDER[1:]))
    print('GAT M1 대비 상대변화(%). 모든 비교·불리한 지표도 CSV에 보존합니다.')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
print('seed49 1시드 — 평균/표준편차·유의성·CLV 배정 귀속 판단 아님')

## 4. 결과 ZIP 다운로드
완료 모형의 원본 결과·1~300epoch 학습 이력·전체 비교·설정·입력 검증 기록을 받습니다. 8개 미완료 시 파일명에 `_partial`을 붙입니다. 재개용 대용량 checkpoint는 ZIP에 넣지 않고 **Drive에 그대로 유지**합니다.

In [ ]:
from google.colab import files
output = screen.export_zip(prepared, '/content/dh_gat_final8_seed49_results.zip')
print(output)
files.download(output)